In [ ]:
# Notebook for learning how to implement Sarsa from scratch

In [ ]:
import gymnasium as gym
import math
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from IPython.display import clear_output

In [ ]:
def make_frozen_lake(render):
    is_slippery = True
    map_name = "4x4"
    render_mode = None
    if render:
        render_mode="rgb_array"
    env = gym.make('FrozenLake-v1', desc=None, map_name=map_name, is_slippery=is_slippery, render_mode=render_mode)
    return env

def make_blackjack(render):
    render_mode = None
    if render:
        render_mode="rgb_array"
    env = gym.make('Blackjack-v1', natural=False, sab=False, render_mode=render_mode)
    return env

def make_taxi(render):
    render_mode = None
    if render:
        render_mode="rgb_array"
    env = gym.make('Taxi-v3', render_mode=render_mode)
    return env

def make_cliffwalking(render):
    render_mode = None
    if render:
        render_mode="rgb_array"
    env = gym.make('CliffWalking-v0', render_mode=render_mode)
    return env

def make_env(render=False):
    return make_frozen_lake(render)

In [ ]:
# Create an environment
env = make_env()
print(env.observation_space.n)
print(env.action_space.n)

In [ ]:
# Evaluate running a greedy policy based on Q
def evaluate(env, Q, render = True):
    done = False
    G_t = 0

    S, info = env.reset()

    if render:
        clear_output(wait=True)
        plt.imshow(env.render())
        plt.show()

    while not done:
        actions = Q[S]
        A = np.argmax(actions)
        S_p, reward, terminated, trunc, info = env.step(A)
        G_t += reward

        S = S_p

        if render:
            clear_output(wait=True)
            plt.imshow(env.render())
            plt.show()

        if terminated or trunc:
            done = True
    
    # return the total return
    return G_t

In [ ]:
start_epsilon = 1.0
gamma = 0.9 # discount factor
lambd = 0.9
alpha = 0.005

epsilon = start_epsilon

def choose_action(actions, epsilon):
    x = np.random.random()
    
    if x < epsilon:
        # Choose action randomly
        action = np.random.randint(0, len(actions))
    else:
        # Choose action greedily
        best_actions = np.argwhere(actions == np.max(actions))
        action_idx = np.random.randint(0, len(best_actions))
        action = best_actions[action_idx][0]
    return action

def step(Q, S, A, E, epsilon):
    # Take action A using e-greedy, observe R,S'
    S_p, reward, terminated, trunc, info = env.step(A)
    
    done = terminated or trunc
    
    # Choose A' from S' using policy derived from Q (eg, e-greedy)
    actions_p = Q[S_p]
    A_p = choose_action(actions_p, epsilon)
    
    delta = reward + gamma * Q[S_p][A_p] - Q[S][A]
    E[S][A] = E[S][A] + 1
    
    for s in range(len(Q)):
        for a in range(len(Q[s])):
            Q[s][a] = Q[s][a] + alpha * delta * E[s][a]
            E[s][a] = gamma * lambd * E[s][a]
    
    return Q, done, S_p, A_p

def episode(Q, epsilon):
    # E(s,a) = 0 for all s element S, a element A(s)
    E = np.zeros((env.observation_space.n, env.action_space.n))
    
    done = False
    S, info = env.reset()
    
    # Choose initial action
    actions = Q[S]
    A = choose_action(actions, epsilon)
    
    # For each step of the episode
    while not done:
        Q, done, S, A = step(Q, S, A, E, epsilon)
    
    return Q

# Initialise Q(s,a) arbitrarily, for all s element S, a element A(s)
Q = np.zeros((env.observation_space.n, env.action_space.n))

num_episodes = 50000
evaluation_run_count = 200
av_returns = []
Q_history = []
Q_history.append(Q.copy())
Q_diffs = []

def log_stats():
    Q_history.append(Q.copy())
    # Calculate the average return over evaluation_run_count
    s = 0
    for _ in range(evaluation_run_count):
        s += evaluate(env, Q, render=False)
    av_return = s / evaluation_run_count
    av_returns.append(av_return)
    # Print average return
    print("Episode: ", episode_idx, "/", num_episodes, ", Epsilon: ", epsilon, ", Average Return: ", av_return)
    # Store how much Q is changing in Q_history
    if len(Q_history) > 1:
        Q_diff = Q_history[-1] - Q_history[-2]
        Q_diff_max = np.max(np.abs(Q_diff))
        Q_diffs.append(Q_diff_max)
        print(f"Q Max Diff: {Q_diff_max:.2f}")

# Learn Q(s,a) over num_episodes
for episode_idx in tqdm(range(num_episodes)):
    Q = episode(Q, epsilon)
    
    if episode_idx % (num_episodes / 10) == 0:
        log_stats()
        epsilon = epsilon * 0.8

# log stats when we've finished
log_stats()

print("Done")

In [ ]:
plt.plot(av_returns)
plt.title("Average Returns")
plt.show()

In [ ]:
plt.plot(Q_diffs)

In [ ]:
# Display the action value function
Q

In [ ]:
env2 = make_env(render=True)

In [ ]:
# Run a single evaluation with rendering so we can see the process work
G = evaluate(env2, Q, render=True)
print("Total Return: ", G)

In [ ]:
s = 0
final_evaluation_run_count = 5000
for _ in range(final_evaluation_run_count):
    s += evaluate(env, Q, render=False)
av_return = s / final_evaluation_run_count
print("Average Return: ", av_return)